In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/raw/spotify_kaggle/dataset.csv")

df = df.drop(columns=["Unnamed: 0"])
df = df.drop_duplicates(subset="track_id", keep="first")
df = df[df["tempo"] > 0]
df = df.dropna(subset=["valence", "energy", "tempo", "track_name", "artists"])

df["artists"]    = df["artists"].str.encode("utf-8", errors="ignore").str.decode("utf-8").str.strip()
df["track_name"] = df["track_name"].str.strip()

def get_action_bucket(valence, energy, tempo):
    v = 0 if valence < 0.33 else 1
    e = 0 if energy  < 0.4  else 1
    t = 0 if tempo   < 100  else 1
    return int(v * 4 + e * 2 + t)

df["action_bucket"] = df.apply(
    lambda r: get_action_bucket(r["valence"], r["energy"], r["tempo"]),
    axis=1
)

keep = ["track_id", "track_name", "artists", "track_genre",
        "valence", "energy", "tempo", "popularity", "action_bucket"]

df[keep].to_csv("../data/processed/music_spotify_clean.csv", index=False)

print(f"Saved {len(df)} tracks")
print(df["action_bucket"].value_counts().sort_index())

Saved 89583 tracks
action_bucket
0     4709
1     5620
2     4312
3    16075
4     2446
5     4891
6    11864
7    39666
Name: count, dtype: int64
